# como para make SQL queries com Claude


In este notebook, we'll explore como para use Claude para generate SQL queries based on natural language questions. We'll set up a teste banco de dados, provide the schema para Claude, e demonstrate como it can understand e translate human language into SQL queries.

## Configuração

primeiro, let's instalar the necessary libraries e Configuração our Anthropic cliente com our API key.

In [ ]:
# Install the necessary libraries
%pip install anthropic

In [2]:
# Import the required libraries
from anthropic import Anthropic
import sqlite3

# Set up the Anthropic API client
client = Anthropic()
MODEL_NAME = "claude-3-opus-20240229"

## Creating a teste banco de dados

We'll criar a teste banco de dados using SQLite e populate it com sample data:

In [3]:
# Connect to the test database (or create it if it doesn't exist)
conn = sqlite3.connect("test_db.db")
cursor = conn.cursor()

# Create a sample table
cursor.execute("""
    CREATE TABLE IF NOT EXISTS employees (
        id INTEGER PRIMARY KEY,
        name TEXT,
        department TEXT,
        salary INTEGER
    )
""")

# Insert sample data
sample_data = [
    (1, "John Doe", "Sales", 50000),
    (2, "Jane Smith", "Engineering", 75000),
    (3, "Mike Johnson", "Sales", 60000),
    (4, "Emily Brown", "Engineering", 80000),
    (5, "David Lee", "Marketing", 55000)
]
cursor.executemany("INSERT INTO employees VALUES (?, ?, ?, ?)", sample_data)
conn.commit()

## Generating SQL Queries com Claude

agora, let's define a função para enviar a natural language question para Claude e get the generated SQL query:

In [8]:
# Define a function to send a query to Claude and get the response
def ask_claude(query, schema):
    prompt = f"""Here is the schema for a database:

{schema}

Given this schema, can you output a SQL query to answer the following question? Only output the SQL query and nothing else.

Question: {query}
"""

    response = client.messages.create(
        model=MODEL_NAME,
        max_tokens=2048,
        messages=[{
            "role": 'user', "content":  prompt
        }]
    )
    return response.content[0].text

We'll retrieve the banco de dados schema e format it as a string:

In [5]:
# Get the database schema
schema = cursor.execute("PRAGMA table_info(employees)").fetchall()
schema_str = "CREATE TABLE EMPLOYEES (\n" + "\n".join([f"{col[1]} {col[2]}" for col in schema]) + "\n)"
print(schema_str)

CREATE TABLE EMPLOYEES (
id INTEGER
name TEXT
department TEXT
salary INTEGER
)


agora, let's provide an example natural language question e enviar it para Claude:

In [9]:
# Example natural language question
question = "What are the names and salaries of employees in the Engineering department?"
# Send the question to Claude and get the SQL query
sql_query = ask_claude(question, schema_str)
print(sql_query)

SELECT name, salary
FROM EMPLOYEES
WHERE department = 'Engineering';


## Executing the Generated SQL Query

Finally, we'll execute the generated SQL query on our teste banco de dados e print the results:

In [10]:
# Execute the SQL query and print the results
results = cursor.execute(sql_query).fetchall()

for row in results:
    print(row)

('Jane Smith', 75000)
('Emily Brown', 80000)


Don't forget para fechar the banco de dados connection quando you're done:

In [11]:
# Close the database connection
conn.close()